# Лабораторна робота 6. Генеративний ШІ в автоматизації
**Конвеєр:** OCR-текст (Лаб. 4) + сутності BERT (Лаб. 5) → LLM (Groq) → Pydantic → арифметична валідація → HITL.

**Перед запуском:** Colab → зліва значок ключа (Secrets) → `groq-api-key` = ваш ключ `gsk_...`, увімкнути *Notebook access*.
Вхідні файли вбудовано в ноутбук (комірка 2): `ocr_result_noisy.txt` = `ocr_result_degraded.txt` з Лаб. 4, `ocr_result_clean.txt`, `ner_extracted.json` з Лаб. 5. Якщо файли вже є в `/content`, вони не перезаписуються.
Запускайте комірки по черзі (**Runtime → Run all**). Скриншоти для звіту робіть з виводу комірок 7–12.

In [ ]:
!pip install -q openai pydantic pandas

In [ ]:
# Комірка 2. Вхідні артефакти Лаб. 4 і Лаб. 5 (вбудовані копії; існуючі файли не перезаписуються)
import os, base64
_EMB = {
    "ocr_result_noisy.txt": "S0wgMyA7JzI1Ckk1CjksOzonIiwKSWUu0YggRGEiCiI00Y8gSQrQmixnJyAgMSAjciIK0LLRhwrQldC00LjRgtGA0LAKMid0MV8KSnUxCkrQmEh0ICwh0LMzCtGP0LNhLNC7bAojINCzemnRgdGPCiM0MSBudXRs0YLQticg0JTQs9Cw0LzRg9GPCjI0bjFzMQpQSmx1awpWSj8K0KLQtjEgI9C4CnszLjExIDsKWCAxRCwKMTEzNDMK0YfQsDU7Cm5sdWEK0YLRhjsnCiIuNTtgClNuc2l0CiPQqNCd0Jog0K5G0LAKMyzRgXcK0JnQvSAxCjYg0LkKSm0uCjExMyw2MwrQo9C+0LsKUnMKItC80LsgRmxheiDQu9GH0YXQugoxNSA1KQrRhdGB0YIzClN1cmlpINCnStC6O3XRggo1LHUKSiI1LDUKJgoqMWPRgiAsCtCcIOKCvdC90Lsg4oKsCiMgOUkKNDouOycK0LjQvSMKSUosCtCwNTsKVDIsbToKJwrRgtC80LrQsNGH0LsKMzc0CiPQvQo1OTQKe3tpIENydGggcj9jaXQgO9GPCjc2NwrRgWvQuNGCOwrQqExJClJGQVVpIDMwCklpJyA7IDMKVWwgODM1ODd4CiMgNjg3NDcyLAoxLCAxMSA/WjYgNCA2CnUK0JYgS0NVT0IyNTQ1Mwo1LgonNTg4MDk0MDU0NwrQkDp1aGYgOyBVbnFmaW47OyArOHvQtiBLClhmbCAhCnV1dSAnCn4KItGB0Lo=",
    "ocr_result_clean.txt": "TApIRUlJIFYmClRJUgoiSmjQoEZpdGYiCk7QtWkKQkRS0KdL0J9F0KbQrApJQtCv0J1VLUBsYdC9awrQpXVqbiAyCkLQpyDQktC40JjQndCV0KbQnQonMzc2NDIxMzYK0J3Qr9CTIHHQm9CY0J0KMzc2NDIxNTI2NTHQnDQKV2xpCltVMTgxICh1cmVubHRsYSDQndCW0LrQntGUbNC5a28KMjM0OTE1MQrQn3VsbNC4ayBOREtBIGl4Z1VDSCBU0JXQmEgKMTY5LFVJSlIKJSAxODks0J9sCtCY0Jcs0JREIDYK0K/QnWvQrwpUSdGBdGxhICDQl9Cd0J3QltCaOzogMjUsNikKMzE5MTUxCtCgddGI0L3QuGsgImxVUm4gJ0TRhdCRVdCh0L0K0KJFSE4K0Kc2OSBVQgrQl9Cd0JjQlmjQnTk7CjExMyw0STAgbgonSzUgCtCp0ZbQvdCwIG50cjthINCn0J3QmNCd0JDQmDogNzUuNjMKMzU4OTczCtCg0YfRiNC90LhrCkRFTk5ERiDQkVXQm9Cf0J/QldC9CiQK0JfQndCY0JbQrDk5ICwgVW4KNzU5NgoxNTk2LDg2IOKCvQrQr9CG0LTQldGP0L1Va9GB0K/RjwrQt9C90LjQttC20LgiCjY1OC40ICkKNzksRzgK0YF10L9uCtGC0YXRhwrQkTEK0J9JSdCSCjJVICwgRmxsIgoxMzEsR3UK0JLQkNCg0KLQmtCvCkpCa0RLQQo30LLQpwo1dSBGTUgKS25mICJn0JHRh9CwawrRiGtNCiI4ODY4INCn0LLQsNC90L4g0Y/Qu9GM0K7RgNCw0L3QutCy0LIK0IbQstCw0L3QsArQqNCf0LDQvWsKQkVMawpQVVM6MTAxCtGH0LVr0LA6IDU5NAo2NjEyMVsyNjE0CkNrb3B1INCnVdCg0J3QkCDQnz/Qr9GC0J3QuNGJ0Y8KNzYiClRhICDQk9Cw0YDQsNCd0KIjINCm0ZbQvdC4INCS0J/Qn9Ce0JRV0JIiIDMwINC00L3RltCyCkRJIDAwNTg3MzYK0KfQldCaICMgMDY3MQoxMiAxMS0yMDI2IDExOjAxIDA4CtCX0J0KS0NVNjgyMzQ5NQp7SCAzdWY5OTQwNTQ3CkFCYW5BQVNVbkFIbkFCWUFVbkdLClhobCAgXSBMVVVMIEt1dyBqIF0gJ2hyenUgdwp+0YLQuNC70L3QuArQlSA1MiBeIDcK0YwKfgp7Qkxia2FsCkZlOwo0CtC30LPQnjE=",
    "ner_extracted.json": "ewogICJkb2N1bWVudF9tZXRhZGF0YSI6IHsKICAgICJzb3VyY2VfZmlsZSI6ICJvY3JfcmVzdWx0X2NsZWFuLnR4dCIsCiAgICAibW9kZWxfbmFtZSI6ICJkc2xpbS9iZXJ0LWJhc2UtTkVSIiwKICAgICJhZ2dyZWdhdGlvbl9zdHJhdGVneSI6ICJzaW1wbGUiLAogICAgInRvdGFsX2VudGl0aWVzX2ZvdW5kIjogMjAKICB9LAogICJlbnRpdGllcyI6IFsKICAgIHsKICAgICAgImVudGl0eV9ncm91cCI6ICJPUkciLAogICAgICAid29yZCI6ICJJIiwKICAgICAgInNjb3JlIjogMC41MDA0LAogICAgICAic3RhcnQiOiA1LAogICAgICAiZW5kIjogNgogICAgfSwKICAgIHsKICAgICAgImVudGl0eV9ncm91cCI6ICJPUkciLAogICAgICAid29yZCI6ICJJIiwKICAgICAgInNjb3JlIjogMC42MDM0LAogICAgICAic3RhcnQiOiAyMCwKICAgICAgImVuZCI6IDIxCiAgICB9LAogICAgewogICAgICAiZW50aXR5X2dyb3VwIjogIkxPQyIsCiAgICAgICJ3b3JkIjogItCSIiwKICAgICAgInNjb3JlIjogMC40NTI3LAogICAgICAic3RhcnQiOiA1OSwKICAgICAgImVuZCI6IDYwCiAgICB9LAogICAgewogICAgICAiZW50aXR5X2dyb3VwIjogIk9SRyIsCiAgICAgICJ3b3JkIjogItCaIiwKICAgICAgInNjb3JlIjogMC41ODA5LAogICAgICAic3RhcnQiOiA4NCwKICAgICAgImVuZCI6IDg1CiAgICB9LAogICAgewogICAgICAiZW50aXR5X2dyb3VwIjogIk9SRyIsCiAgICAgICJ3b3JkIjogItCcIiwKICAgICAgInNjb3JlIjogMC40ODIzLAogICAgICAic3RhcnQiOiA5MywKICAgICAgImVuZCI6IDk0CiAgICB9LAogICAgewogICAgICAiZW50aXR5X2dyb3VwIjogIk9SRyIsCiAgICAgICJ3b3JkIjogItCUIiwKICAgICAgInNjb3JlIjogMC45MzYyLAogICAgICAic3RhcnQiOiAxMTIsCiAgICAgICJlbmQiOiAxMTMKICAgIH0sCiAgICB7CiAgICAgICJlbnRpdHlfZ3JvdXAiOiAiT1JHIiwKICAgICAgIndvcmQiOiAi0J8iLAogICAgICAic2NvcmUiOiAwLjcwMzcsCiAgICAgICJzdGFydCI6IDExNCwKICAgICAgImVuZCI6IDExNQogICAgfSwKICAgIHsKICAgICAgImVudGl0eV9ncm91cCI6ICJPUkciLAogICAgICAid29yZCI6ICJJSUgiLAogICAgICAic2NvcmUiOiAwLjc3NjYsCiAgICAgICJzdGFydCI6IDEyOSwKICAgICAgImVuZCI6IDEzMgogICAgfSwKICAgIHsKICAgICAgImVudGl0eV9ncm91cCI6ICJPUkciLAogICAgICAid29yZCI6ICJVIiwKICAgICAgInNjb3JlIjogMC44MjMyLAogICAgICAic3RhcnQiOiAyNTYsCiAgICAgICJlbmQiOiAyNTcKICAgIH0sCiAgICB7CiAgICAgICJlbnRpdHlfZ3JvdXAiOiAiTE9DIiwKICAgICAgIndvcmQiOiAi0KYiLAogICAgICAic2NvcmUiOiAwLjM4OSwKICAgICAgInN0YXJ0IjogMjg0LAogICAgICAiZW5kIjogMjg1CiAgICB9LAogICAgewogICAgICAiZW50aXR5X2dyb3VwIjogIk9SRyIsCiAgICAgICJ3b3JkIjogIlUiLAogICAgICAic2NvcmUiOiAwLjQ3OTQsCiAgICAgICJzdGFydCI6IDM0MCwKICAgICAgImVuZCI6IDM0MQogICAgfSwKICAgIHsKICAgICAgImVudGl0eV9ncm91cCI6ICJPUkciLAogICAgICAid29yZCI6ICJVIiwKICAgICAgInNjb3JlIjogMC43NTQ1LAogICAgICAic3RhcnQiOiAzODcsCiAgICAgICJlbmQiOiAzODgKICAgIH0sCiAgICB7CiAgICAgICJlbnRpdHlfZ3JvdXAiOiAiUEVSIiwKICAgICAgIndvcmQiOiAi0IYiLAogICAgICAic2NvcmUiOiAwLjU4ODEsCiAgICAgICJzdGFydCI6IDUxNywKICAgICAgImVuZCI6IDUxOAogICAgfSwKICAgIHsKICAgICAgImVudGl0eV9ncm91cCI6ICJQRVIiLAogICAgICAid29yZCI6ICJDa28iLAogICAgICAic2NvcmUiOiAwLjc4MDcsCiAgICAgICJzdGFydCI6IDU2NCwKICAgICAgImVuZCI6IDU2NwogICAgfSwKICAgIHsKICAgICAgImVudGl0eV9ncm91cCI6ICJQRVIiLAogICAgICAid29yZCI6ICLQpyIsCiAgICAgICJzY29yZSI6IDAuOTA4LAogICAgICAic3RhcnQiOiA1NzAsCiAgICAgICJlbmQiOiA1NzEKICAgIH0sCiAgICB7CiAgICAgICJlbnRpdHlfZ3JvdXAiOiAiUEVSIiwKICAgICAgIndvcmQiOiAiVGEiLAogICAgICAic2NvcmUiOiAwLjUwMzcsCiAgICAgICJzdGFydCI6IDU5NCwKICAgICAgImVuZCI6IDU5NgogICAgfSwKICAgIHsKICAgICAgImVudGl0eV9ncm91cCI6ICJPUkciLAogICAgICAid29yZCI6ICLQkyIsCiAgICAgICJzY29yZSI6IDAuNjIxMywKICAgICAgInN0YXJ0IjogNTk3LAogICAgICAiZW5kIjogNTk4CiAgICB9LAogICAgewogICAgICAiZW50aXR5X2dyb3VwIjogIk9SRyIsCiAgICAgICJ3b3JkIjogItCoIiwKICAgICAgInNjb3JlIjogMC43NDE5LAogICAgICAic3RhcnQiOiA2MDUsCiAgICAgICJlbmQiOiA2MDYKICAgIH0sCiAgICB7CiAgICAgICJlbnRpdHlfZ3JvdXAiOiAiT1JHIiwKICAgICAgIndvcmQiOiAiQUIiLAogICAgICAic2NvcmUiOiAwLjY3NDgsCiAgICAgICJzdGFydCI6IDcxNCwKICAgICAgImVuZCI6IDcxNgogICAgfSwKICAgIHsKICAgICAgImVudGl0eV9ncm91cCI6ICJPUkciLAogICAgICAid29yZCI6ICJISCIsCiAgICAgICJzY29yZSI6IDAuNjIxLAogICAgICAic3RhcnQiOiA3MjMsCiAgICAgICJlbmQiOiA3MjUKICAgIH0KICBdCn0=",
}
for _name, _b in _EMB.items():
    if not os.path.exists(_name):
        with open(_name, "wb") as _f:
            _f.write(base64.b64decode(_b))
        print("[+] створено", _name)
    else:
        print("[=] вже існує", _name)

In [ ]:
# Крок 1. Pydantic-схема (Optional-поля витримують втрату чисел через шум OCR)
from pydantic import BaseModel, Field
from typing import List, Optional

class InvoiceItem(BaseModel):
    name: str = Field(description="Повна назва знайденого товару/послуги")
    quantity: float = Field(description="Кількість куплених одиниць", default=1.0)
    price: Optional[float] = Field(description="Ціна за одиницю товару", default=None)
    total: Optional[float] = Field(description="Підсумкова вартість даної товарної позиції", default=None)

class ExtractedInvoice(BaseModel):
    reasoning: str = Field(
        description=(
            "Chain-of-Thought міркування: 1) виправлення помилок OCR у назві закладу; "
            "2) порівняння з токенними сутностями Лаб. 5; 3) виявлення та групування товарів; "
            "4) перевірка математичної суми позицій відносно підсумкового TOTAL."
        )
    )
    merchant_name: Optional[str] = Field(description="Назва торгової точки або мережі", default=None)
    date: Optional[str] = Field(description="Дата транзакції у форматі YYYY-MM-DD", default=None)
    items: List[InvoiceItem] = Field(description="Список розпізнаних позицій", default_factory=list)
    total_amount: Optional[float] = Field(description="Підсумкова сума до сплати (TOTAL)", default=None)

print("[+] Схему ExtractedInvoice скомпільовано.")

In [ ]:
# Крок 2. Клієнт Groq + вибір моделі
import os, json
import pandas as pd
from openai import OpenAI

try:
    from google.colab import userdata
    raw_key = userdata.get("groq-api-key")
except Exception:
    raw_key = os.environ.get("GROQ_API_KEY")
if not raw_key:
    raise ValueError("Секрет 'groq-api-key' не знайдено!")

client_groq = OpenAI(api_key=raw_key.strip(), base_url="https://api.groq.com/openai/v1")

# Модель із методички; якщо її немає у вашому акаунті - беремо іншу зі списку доступних
PREFERRED = "qwen/qwen3.8-27b"
available = [m.id for m in client_groq.models.list().data]
print("Доступні моделі:", available)
if PREFERRED in available:
    MODEL = PREFERRED
else:
    skip = ("whisper", "guard", "tts", "orpheus", "playai", "distil")
    cand = [m for m in available if not any(s in m.lower() for s in skip)]
    MODEL = next((m for m in cand if "qwen" in m.lower()), None) or next((m for m in cand if "llama" in m.lower()), None) or cand[0]
print("\n[+] Обрана модель:", MODEL, "  (запишіть у звіт; ця ж модель піде в n8n)")

In [ ]:
# Читання вхідних даних
with open("ocr_result_noisy.txt", "r", encoding="utf-8") as f:
    raw_ocr_text = f.read().strip()
with open("ocr_result_clean.txt", "r", encoding="utf-8") as f:
    clean_ocr_text = f.read().strip()
with open("ner_extracted.json", "r", encoding="utf-8") as f:
    bert_entities = json.load(f)

print("[+] Артефакт Лаб. 4 (ocr_result_noisy.txt):", len(raw_ocr_text), "символів")
print("[+] Артефакт Лаб. 5 (ner_extracted.json):", bert_entities["document_metadata"]["total_entities_found"], "сутностей")
print("\n--- Фрагмент зашумленого тексту ---")
print(raw_ocr_text[:400])
print("\n--- Сутності, розпізнані моделлю BERT (Лаб. 5) ---")
print(json.dumps(bert_entities, ensure_ascii=False, indent=2)[:1500], "...")

In [ ]:
# Допоміжні функції (чисті, без залежності від API)
import re, hashlib

def extract_json(text):
    # Дістає JSON-об'єкт з відповіді: прибирає <think>-блоки та markdown-огорожі
    if text is None:
        raise ValueError("порожня відповідь моделі")
    t = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S).strip()
    a, b = t.find("{"), t.rfind("}")
    if a == -1 or b == -1:
        raise ValueError("JSON-об'єкт не знайдено у відповіді")
    return json.loads(t[a:b + 1])

def validate_invoice(d, tol=0.05):
    # Детермінована арифметична й бізнес-валідація (Крок 6)
    issues = []
    items = d.get("items") or []
    calc = 0.0
    for it in items:
        t = it.get("total")
        if t is None and it.get("price") is not None:
            t = it["price"] * (it.get("quantity") or 1.0)
        if t is None:
            issues.append("позиція «%s» без суми" % it.get("name"))
        else:
            calc += t
    doc_total = d.get("total_amount")
    if not d.get("merchant_name"):
        issues.append("не визначено назву торгової точки")
    if not items:
        issues.append("список позицій порожній")
    if doc_total is None or doc_total <= 0:
        issues.append("не визначено підсумкову суму (TOTAL)")
        doc_total = 0.0
    disc = round(abs(calc - doc_total), 2)
    if doc_total > 0 and items and disc > tol:
        issues.append("сума позицій %.2f не збігається з TOTAL %.2f" % (calc, doc_total))
    return {"is_valid": not issues, "calc_sum": round(calc, 2), "doc_total": doc_total,
            "discrepancy": disc, "issues": issues}

# Еталон (прочитано з фото чека JYSK, 12-11-2026)
GROUND_TRUTH = {
    "merchant_aliases": ["jysk", "юск"],
    "date": "2026-11-12",
    "total_amount": 5200.00,
    "item_name_aliases": ["svanek", "стіл"],
    "item_price": 11999.00,      # ціна за одиницю до знижки
}
FIELDS = ["Назва торгової точки", "Дата", "Підсумкова сума", "Назва товару", "Ціна товару"]

def _has(s, aliases):
    s = (s or "").lower()
    return any(a in s for a in aliases)

def score_llm(d):
    it = (d.get("items") or [{}])[0] if d.get("items") else {}
    return [
        _has(d.get("merchant_name"), GROUND_TRUTH["merchant_aliases"]),
        d.get("date") == GROUND_TRUTH["date"],
        d.get("total_amount") is not None and abs(d["total_amount"] - GROUND_TRUTH["total_amount"]) < 0.01,
        _has(it.get("name"), GROUND_TRUTH["item_name_aliases"]),
        it.get("price") is not None and abs(it["price"] - GROUND_TRUTH["item_price"]) < 0.01,
    ]

def score_bert(ents):
    # BERT-NER (dslim) має лише класи PER/ORG/LOC/MISC: дат, сум і цін не виділяє взагалі
    words = " ".join(e["word"] for e in ents["entities"] if e["entity_group"] in ("ORG", "LOC", "MISC"))
    return [_has(words, GROUND_TRUTH["merchant_aliases"]), False, False, False, False]

def signature(d):
    # Відбиток ключових полів - для підрахунку унікальних відповідей між запусками
    key = json.dumps([d.get("merchant_name"), d.get("date"), d.get("total_amount"),
                      [(i.get("name"), i.get("price"), i.get("total")) for i in d.get("items", [])]],
                     ensure_ascii=False, sort_keys=True)
    return hashlib.md5(key.encode()).hexdigest()[:8]
print("[+] Допоміжні функції готові.")

In [ ]:
# Крок 3. Temperature = 1.0, без схеми й без обмежень (демонстрація ризику галюцинацій)
print("[*] Тест генерації: Temperature = 1.0 (без обмеження схеми), 3 запуски\n")
chaos_outputs = []
for i in range(3):
    try:
        r = client_groq.chat.completions.create(
            model=MODEL, temperature=1.0,
            messages=[{"role": "system", "content": "Витягни дані з цього чека у JSON."},
                      {"role": "user", "content": raw_ocr_text}])
        out = r.choices[0].message.content
    except Exception as e:
        out = "[ПОМИЛКА API] %s" % e
    chaos_outputs.append(out)
    print("--- Запуск %d (T=1.0) ---" % (i + 1))
    print(out[:1200])
    print()
print("Унікальних відповідей з 3:", len(set(chaos_outputs)))

In [ ]:
# Крок 4. Трирівневий промпт: System + Few-Shot + Chain-of-Thought, обмеження json_object
schema = ExtractedInvoice.model_json_schema()

system_prompt = (
    "Ти — експертний автономний модуль інтелектуального вилучення даних з документів. "
    "Тобі надано: 1) сирий текст чека після OCR (Лаб. 4); 2) попередньо розпізнані моделлю BERT сутності (Лаб. 5). "
    "Твоє завдання — зіставити ці джерела, виправити шуми розпізнавання, згрупувати товари з їхніми цінами "
    "та повернути СТРОГО валідний JSON за наданою схемою.\n"
    "Контракт схеми: " + json.dumps(schema, ensure_ascii=False) + "\n"
    "ВИМОГИ:\n"
    "- Поверни ТІЛЬКИ чистий JSON-об'єкт без маркдаун-тегів.\n"
    "- Обов'язково заповни поле reasoning, де опиши хід думок та звірку цін.\n"
    "- Використовуй точні імена ключів: reasoning, merchant_name, date, items, total_amount.\n"
    "- Якщо значення неможливо надійно відновити з тексту, став null (для items — порожній список). "
    "НЕ вигадуй дані, яких немає в джерелах."
)

few_shot_example = (
    "\n\nПРИКЛАД ОБРОБКИ:\n"
    "Текст: 'ATB-Markt 12.05.2025 HLIB 24.00 MOLOKO 36.00 SUMA: 60.00'\n"
    "Відповідь: {\n"
    ' "reasoning": "Виправлено назву на ATB-Market; дата 2025-05-12; сума позицій 24+36=60 відповідає підсумку 60.",\n'
    ' "merchant_name": "ATB-Market",\n'
    ' "date": "2025-05-12",\n'
    ' "items": [{"name": "HLIB", "quantity": 1.0, "price": 24.0, "total": 24.0}, {"name": "MOLOKO", "quantity": 1.0, "price": 36.0, "total": 36.0}],\n'
    ' "total_amount": 60.0\n'
    "}"
)
system_prompt = system_prompt + few_shot_example

# Компактна версія сутностей BERT для контексту
bert_compact = [{"group": e["entity_group"], "word": e["word"], "score": e["score"]} for e in bert_entities["entities"]]

def extract_invoice(ocr_text, temperature):
    # Один виклик LLM -> dict з сирою відповіддю, розібраним JSON і Pydantic-об'єктом
    res = {"raw": None, "data": None, "model": None, "error": None}
    try:
        r = client_groq.chat.completions.create(
            model=MODEL, temperature=temperature,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": (
                    "=== ДЖЕРЕЛО 1: Сирий текст з Лабораторної № 4 (OCR) ===\n%s\n\n"
                    "=== ДЖЕРЕЛО 2: Сутності BERT з Лабораторної № 5 ===\n%s"
                ) % (ocr_text, json.dumps(bert_compact, ensure_ascii=False))},
            ],
            response_format={"type": "json_object"},
        )
        res["raw"] = r.choices[0].message.content
        res["data"] = extract_json(res["raw"])
        res["model"] = ExtractedInvoice.model_validate(res["data"])   # валідація контракту Pydantic
    except Exception as e:
        res["error"] = "%s: %s" % (type(e).__name__, str(e)[:300])
    return res

print("[*] Детермінована екстракція: Temperature = 0.0")
main = extract_invoice(raw_ocr_text, 0.0)
if main["error"]:
    print("[!] Помилка:", main["error"])
    print(main["raw"])
else:
    llm_data = main["model"]
    print("=================== ХІД ДУМОК МОДЕЛІ (Chain-of-Thought) ===================")
    print(llm_data.reasoning)
    print("==========================================================================")
    print("Магазин:", llm_data.merchant_name)
    print("Дата:", llm_data.date)
    print("Загальна сума:", llm_data.total_amount)
    print("\nЗнайдені позиції:")
    for it in llm_data.items:
        print(f" - {it.name:<25} | К-сть: {it.quantity:<3} | Ціна: {it.price} | Разом: {it.total}")
    if not llm_data.items:
        print(" (позицій не знайдено)")

In [ ]:
# Завдання 3. Вплив Temperature: T=1.0 проти T=0.0 (трирівневий промпт + json_object), по 3 запуски
rows = []
runs_store = {}
for T in (1.0, 0.0):
    runs = [extract_invoice(raw_ocr_text, T) for _ in range(3)]
    runs_store[T] = runs
    ok_json = [r for r in runs if r["data"] is not None]
    ok_schema = [r for r in runs if r["model"] is not None]
    sigs = [signature(r["data"]) for r in ok_json]
    acc = [sum(score_llm(r["data"])) / 5 for r in ok_json]
    rows.append({
        "Temperature": T,
        "Запусків": len(runs),
        "Валідний JSON": "%d/3" % len(ok_json),
        "Валідна схема Pydantic": "%d/3" % len(ok_schema),
        "Унікальних відповідей": len(set(sigs)) if sigs else 0,
        "Середня точність полів": ("%.0f%%" % (100 * sum(acc) / len(acc))) if acc else "н/д",
        "Помилки API/парсингу": "; ".join(sorted({r["error"][:60] for r in runs if r["error"]})) or "—",
    })
df_temp = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
display(df_temp)

In [ ]:
# Крок 5. Порівняння BERT (Лаб. 5) та LLM (Лаб. 6) з еталоном з фото чека
llm_dict = main["data"] or {}
bert_ok = score_bert(bert_entities)
llm_ok = score_llm(llm_dict) if llm_dict else [False] * 5

ents = bert_entities["entities"]
bert_words = ", ".join(e["word"] for e in ents if e["entity_group"] in ("ORG", "LOC", "MISC"))[:60]
llm_items = llm_dict.get("items") or [{}]
first = llm_items[0] if llm_items else {}

truth = ["JYSK (ТОВ «ЮСК УКРАЇНА»)", "2026-11-12", "5200.00", "Стіл регульований SVANEK", "11999.00"]
bert_vals = [bert_words + "…", "— (клас DATE відсутній)", "— (клас TOTAL відсутній)", "— (клас ITEM відсутній)", "— (клас PRICE відсутній)"]
llm_vals = [str(llm_dict.get("merchant_name")), str(llm_dict.get("date")), str(llm_dict.get("total_amount")),
            str(first.get("name")), str(first.get("price"))]

df_benchmark = pd.DataFrame({
    "Критерій / Поле": FIELDS,
    "Еталон (фото чека)": truth,
    "BERT-NER (Лаб. 5)": bert_vals,
    "BERT ✓": ["✓" if x else "✗" for x in bert_ok],
    "LLM Groq (Лаб. 6)": llm_vals,
    "LLM ✓": ["✓" if x else "✗" for x in llm_ok],
})
display(df_benchmark)

short = sum(len(e["word"]) <= 2 for e in ents)
print("\nBERT: сутностей %d, з них односимвольних/двосимвольних шумових: %d (%.0f%%); середній score %.2f"
      % (len(ents), short, 100 * short / len(ents), sum(e["score"] for e in ents) / len(ents)))
print("Точність за 5 полями: BERT = %d/5, LLM = %d/5" % (sum(bert_ok), sum(llm_ok)))

In [ ]:
# Крок 6. Програмна бізнес-валідація (Cross-Validation)
print("[*] Детермінована перевірка арифметичного балансу")
val = validate_invoice(llm_dict)
print("Розрахована сума окремих позицій: %.2f" % val["calc_sum"])
print("Сума, заявлена у чеку (TOTAL):     %.2f" % val["doc_total"])
print("Абсолютна розбіжність:             %.2f" % val["discrepancy"])
if val["is_valid"]:
    print("\n[✓] СТАТУС: VALIDATION PASSED. Транзакція консистентна -> Маршрутизація в ERP/Таблицю.")
else:
    print("\n[!] СТАТУС: VALIDATION FAILED -> Маршрутизація оператору (Human-in-the-Loop).")
    for i in val["issues"]:
        print("   -", i)

In [ ]:
# Додатково: той самий конвеєр на тексті ocr_result_clean.txt (інша якість OCR, Лаб. 4)
clean_run = extract_invoice(clean_ocr_text, 0.0)
cd = clean_run["data"] or {}
cv = validate_invoice(cd) if cd else None
cmp_rows = []
for label, d, v in (("noisy (degraded)", llm_dict, val), ("clean", cd, cv)):
    sc = score_llm(d) if d else [False] * 5
    cmp_rows.append({"Вхід OCR": label,
                     "merchant": (d or {}).get("merchant_name"),
                     "date": (d or {}).get("date"),
                     "total": (d or {}).get("total_amount"),
                     "позицій": len((d or {}).get("items") or []),
                     "точність полів": "%d/5" % sum(sc),
                     "статус": ("PASSED" if v and v["is_valid"] else "FAILED -> HITL")})
display(pd.DataFrame(cmp_rows))
if clean_run["error"]:
    print("Помилка на clean:", clean_run["error"])

In [ ]:
# Збереження артефактів (для звіту, GitHub та n8n)
with open("llm_result.json", "w", encoding="utf-8") as f:
    json.dump({"model": MODEL, "temperature": 0.0, "data": llm_dict, "validation": val}, f, ensure_ascii=False, indent=2)
df_benchmark.to_csv("comparison_bert_vs_llm.csv", index=False, encoding="utf-8-sig")
df_temp.to_csv("temperature_experiment.csv", index=False, encoding="utf-8-sig")
print("[+] Збережено: llm_result.json, comparison_bert_vs_llm.csv, temperature_experiment.csv")
try:
    from google.colab import files
    for n in ("llm_result.json", "comparison_bert_vs_llm.csv", "temperature_experiment.csv"):
        files.download(n)
except Exception:
    pass